# Simon's Algorithm

## The closest precursor to Shor: an *exponential* speedup against any classical algorithm (even randomised), using period-finding by Fourier sampling. Mastering Simon makes Shor much easier to read.

# 1. The problem

## Given an oracle for $f : \{0,1\}^n \to \{0,1\}^n$ with the promise that there exists a hidden $s \in \{0,1\}^n$ such that

$$ \Large  f(x) = f(y) \;\iff\; x = y \text{ or } x = y \oplus s, $$

## find $s$.

## **Classically**: requires $\Theta(2^{n/2})$ queries (by birthday-style arguments). **Quantumly**: $O(n)$ queries.

# 2. The algorithm

## 1. Initialise $n$ input qubits in $|0\rangle^{\otimes n}$ and $n$ output qubits in $|0\rangle^{\otimes n}$.
## 2. Apply $H^{\otimes n}$ to the input register.
## 3. Apply $U_f$.
## 4. Apply $H^{\otimes n}$ to the input register.
## 5. Measure the input register.

## Each run yields a random $y$ such that $s \cdot y \equiv 0 \pmod 2$. Repeat $O(n)$ times to gather $n - 1$ linearly independent such $y$, then solve the linear system over $\mathbb F_2$ to recover $s$.

# 3. Why it works

## After step 2 the input register is uniform: $\frac{1}{\sqrt{2^n}}\sum_x |x\rangle$.

## After step 3 the joint state is $\frac{1}{\sqrt{2^n}}\sum_x |x\rangle|f(x)\rangle$.

## Suppose we now measure the output register and get some value $z$. The input register collapses to a superposition of the *pre-images* of $z$ — which by the promise is $\{x_0, x_0 \oplus s\}$ for some $x_0$. So the input register is

$$ \Large  \tfrac{1}{\sqrt 2}(|x_0\rangle + |x_0 \oplus s\rangle). $$

## Apply $H^{\otimes n}$:

$$ \Large  \frac{1}{\sqrt{2^{n+1}}} \sum_y \big( (-1)^{x_0 \cdot y} + (-1)^{(x_0 \oplus s) \cdot y} \big) |y\rangle = \frac{1}{\sqrt{2^{n+1}}} \sum_y (-1)^{x_0 \cdot y} (1 + (-1)^{s \cdot y}) |y\rangle. $$

## Only those $y$ with $s \cdot y \equiv 0$ survive — the rest are killed by destructive interference. So each measurement outputs a uniformly random $y$ orthogonal to $s$ (over $\mathbb F_2$). After $\Theta(n)$ runs we expect to gather $n - 1$ independent equations, and Gaussian elimination delivers $s$.

In [ ]:
# A Qiskit implementation for n = 3 and a chosen secret
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

def simon_oracle(s):
    n = len(s)
    qc = QuantumCircuit(2*n)
    # copy input to output via CNOTs
    for i in range(n):
        qc.cx(i, n + i)
    # XOR the output with s, conditioned on a fixed input bit (any will do)
    j = s.find('1')
    if j != -1:
        for i, bit in enumerate(s):
            if bit == '1':
                qc.cx(j, n + i)
    return qc

def run_simon(s, shots=1024):
    n = len(s)
    qc = QuantumCircuit(2*n, n)
    qc.h(range(n))
    qc.compose(simon_oracle(s), inplace=True)
    qc.h(range(n))
    qc.measure(range(n), range(n))
    sim = AerSimulator()
    return sim.run(transpile(qc, sim), shots=shots).result().get_counts()

secret = '110'
counts = run_simon(secret)
print('Outcomes (each y satisfies s . y = 0):')
for y in counts:
    print(f'  y = {y}')

# 4. Why this matters for Shor

## Simon is **period finding over $(\mathbb{Z}/2)^n$**. Replace this group with $\mathbb{Z}/N$, the Hadamard with the Quantum Fourier Transform, and you get the period-finding subroutine that powers Shor's algorithm. The structure is identical:

## - prepare a uniform superposition of inputs,
## - apply the oracle to create entanglement between inputs and outputs,
## - Fourier-transform the input register,
## - measure and read off information about the hidden period.

# Recap

## - $f$ hides a period $s$ in $(\mathbb{Z}/2)^n$: $f(x) = f(x \oplus s)$.
## - Quantum: $O(n)$ queries; classical: $\Omega(2^{n/2})$.
## - Each run produces a random $y$ orthogonal to $s$; linear-algebra to recover $s$.
## - Direct blueprint for Shor's algorithm.

## **End of chapter 4.** Next: Grover's search.